In [ ]:
# ImageToolkit.ipynb - Theory & Practice for Image Processing
"""
Jupyter Notebook: Image Processing Theory and Practice
===============================================

This notebook provides step-by-step theory and hands-on practice for fundamental 
image processing operations using OpenCV and Python.

Table of Contents:
1. Setup and Fundamentals
2. Image Representation & Color Spaces
3. Filtering & Noise Reduction
4. Edge Detection
5. Morphological Operations
6. Image Enhancement
7. Frequency Domain Processing
8. Practical Applications
"""

# =============================================================================
#  1: Setup and Imports
# =============================================================================

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
from scipy import ndimage
import warnings
warnings.filterwarnings('ignore')

# Set matplotlib style
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)

print("Libraries imported successfully!")
print("OpenCV Version:", cv2.__version__)
print("NumPy Version:", np.__version__)

# =============================================================================
#2: Helper Functions
# =============================================================================

def display_images(images, titles, figsize=(15, 5), cmap='gray'):
    """Display multiple images side by side"""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    
    for i in range(n):
        if len(images[i].shape) == 3:
            axes[i].imshow(cv2.cvtColor(images[i], cv2.COLOR_BGR2RGB))
        else:
            axes[i].imshow(images[i], cmap=cmap)
        axes[i].set_title(titles[i])
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

def create_sample_image():
    """Create a sample image for testing"""
    img = np.zeros((200, 200, 3), dtype=np.uint8)
    cv2.circle(img, (100, 100), 50, (255, 255, 255), -1)
    cv2.rectangle(img, (50, 50), (150, 150), (128, 128, 128), 2)
    return img

def add_noise(image, noise_type='gaussian'):
    """Add different types of noise to image"""
    if noise_type == 'gaussian':
        noise = np.random.normal(0, 25, image.shape).astype(np.uint8)
        return cv2.add(image, noise)
    elif noise_type == 'salt_pepper':
        noise = np.random.random(image.shape[:2])
        image_copy = image.copy()
        image_copy[noise < 0.05] = 0
        image_copy[noise > 0.95] = 255
        return image_copy

# =============================================================================
# 3: Chapter 1 - Image Fundamentals
# =============================================================================

print("=" * 60)
print("CHAPTER 1: IMAGE FUNDAMENTALS")
print("=" * 60)

# Create sample image
sample_img = create_sample_image()

print("Theory:")
print("-------")
print("• Digital images are 2D arrays of pixels")
print("• Each pixel has intensity values (0-255 for 8-bit)")
print("• Color images have 3 channels: Red, Green, Blue")
print("• Image shape: (height, width, channels)")

print(f"\nSample Image Properties:")
print(f"Shape: {sample_img.shape}")
print(f"Data type: {sample_img.dtype}")
print(f"Min value: {np.min(sample_img)}")
print(f"Max value: {np.max(sample_img)}")

# Display original image
display_images([sample_img], ["Sample Image"])

# Task 1: Image Analysis
print("\n TASK 1: Analyze different image properties")

def analyze_image(img):
    """Comprehensive image analysis"""
    print(f"Image Shape: {img.shape}")
    print(f"Total Pixels: {img.size}")
    print(f"Data Type: {img.dtype}")
    print(f"Memory Usage: {img.nbytes} bytes")
    
    if len(img.shape) == 3:
        print("Color Channels:")
        for i, channel in enumerate(['Blue', 'Green', 'Red']):
            print(f"  {channel} - Mean: {np.mean(img[:,:,i]):.2f}, Std: {np.std(img[:,:,i]):.2f}")
    else:
        print(f"Grayscale - Mean: {np.mean(img):.2f}, Std: {np.std(img):.2f}")

analyze_image(sample_img)

# =============================================================================
# 4: Chapter 2 - Color Space Conversions
# =============================================================================

print("\n" + "=" * 60)
print("CHAPTER 2: COLOR SPACE CONVERSIONS")
print("=" * 60)

print("Theory:")
print("-------")
print("• RGB: Red, Green, Blue - additive color model")
print("• HSV: Hue, Saturation, Value - intuitive for color selection")
print("• YCrCb: Luminance (Y) and Chrominance (Cr, Cb) - used in compression")
print("• Grayscale: Single channel representing intensity")

# Color space conversions
rgb_img = sample_img.copy()
gray_img = cv2.cvtColor(rgb_img, cv2.COLOR_BGR2GRAY)
hsv_img = cv2.cvtColor(rgb_img, cv2.COLOR_BGR2HSV)
ycrcb_img = cv2.cvtColor(rgb_img, cv2.COLOR_BGR2YCrCb)

display_images([rgb_img, gray_img, hsv_img, ycrcb_img], 
               ['RGB', 'Grayscale', 'HSV', 'YCrCb'])

# Task 2: Manual color conversion
print("\n TASK 2: Implement manual RGB to Grayscale conversion")

def rgb_to_gray_manual(img):
    """Manual RGB to grayscale conversion using luminance formula"""
    # Luminance formula: Y = 0.299*R + 0.587*G + 0.114*B
    return np.uint8(0.299 * img[:,:,2] + 0.587 * img[:,:,1] + 0.114 * img[:,:,0])

manual_gray = rgb_to_gray_manual(rgb_img)
opencv_gray = cv2.cvtColor(rgb_img, cv2.COLOR_BGR2GRAY)

display_images([manual_gray, opencv_gray], ['Manual Conversion', 'OpenCV Conversion'])

# =============================================================================
#  5: Chapter 3 - Filtering and Noise Reduction
# =============================================================================

print("\n" + "=" * 60)
print("CHAPTER 3: FILTERING AND NOISE REDUCTION")
print("=" * 60)

print("Theory:")
print("-------")
print("• Filtering: Convolution operation with kernels")
print("• Gaussian filter: Reduces noise, preserves edges poorly")
print("• Median filter: Effective for salt-and-pepper noise")
print("• Bilateral filter: Edge-preserving noise reduction")

# Create noisy image
noisy_gaussian = add_noise(gray_img, 'gaussian')
noisy_sp = add_noise(gray_img, 'salt_pepper')

display_images([gray_img, noisy_gaussian, noisy_sp], 
               ['Original', 'Gaussian Noise', 'Salt & Pepper Noise'])

# Apply different filters
gaussian_filtered = cv2.GaussianBlur(noisy_gaussian, (5, 5), 1.0)
median_filtered = cv2.medianBlur(noisy_sp, 5)
bilateral_filtered = cv2.bilateralFilter(noisy_gaussian, 9, 75, 75)

display_images([gaussian_filtered, median_filtered, bilateral_filtered],
               ['Gaussian Filter', 'Median Filter', 'Bilateral Filter'])

# Task 3: Custom kernel implementation
print("\n TASK 3: Implement custom convolution")

def custom_filter(img, kernel):
    """Apply custom kernel using convolution"""
    return cv2.filter2D(img, -1, kernel)

# Define custom kernels
sharpen_kernel = np.array([[-1, -1, -1],
                          [-1,  9, -1],
                          [-1, -1, -1]])

edge_kernel = np.array([[-1, -1, -1],
                       [-1,  8, -1],
                       [-1, -1, -1]])

sharpened = custom_filter(gray_img, sharpen_kernel)
edges = custom_filter(gray_img, edge_kernel)

display_images([gray_img, sharpened, edges], 
               ['Original', 'Sharpened', 'Edge Enhanced'])

# =============================================================================
# 6: Chapter 4 - Edge Detection
# =============================================================================

print("\n" + "=" * 60)
print("CHAPTER 4: EDGE DETECTION")
print("=" * 60)

print("Theory:")
print("-------")
print("• Edges: Rapid changes in intensity")
print("• Sobel: First derivative approximation")
print("• Laplacian: Second derivative, sensitive to noise")
print("• Canny: Multi-stage algorithm (optimal edge detector)")
print("\nCanny Algorithm Steps:")
print("1. Gaussian smoothing")
print("2. Gradient calculation")
print("3. Non-maximum suppression")
print("4. Double thresholding")
print("5. Edge tracking by hysteresis")

# Sobel edge detection
sobelx = cv2.Sobel(gray_img, cv2.CV_64F, 1, 0, ksize=3)
sobely = cv2.Sobel(gray_img, cv2.CV_64F, 0, 1, ksize=3)
sobel_combined = np.sqrt(sobelx**2 + sobely**2)

# Laplacian edge detection
laplacian = cv2.Laplacian(gray_img, cv2.CV_64F)

# Canny edge detection
canny = cv2.Canny(gray_img, 50, 150)

display_images([sobelx, sobely, sobel_combined, laplacian, canny],
               ['Sobel X', 'Sobel Y', 'Sobel Combined', 'Laplacian', 'Canny'])

# Task 4: Compare edge detection methods
print("\n TASK 4: Compare edge detection performance")

def compare_edge_detectors(img):
    """Compare different edge detection methods"""
    methods = {
        'Sobel': np.uint8(np.sqrt(cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)**2 + 
                                 cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)**2)),
        'Laplacian': np.uint8(np.absolute(cv2.Laplacian(img, cv2.CV_64F))),
        'Canny': cv2.Canny(img, 50, 150)
    }
    
    images = [img] + list(methods.values())
    titles = ['Original'] + list(methods.keys())
    display_images(images, titles)

compare_edge_detectors(gray_img)

# =============================================================================
# 7: Chapter 5 - Morphological Operations
# =============================================================================

print("\n" + "=" * 60)
print("CHAPTER 5: MORPHOLOGICAL OPERATIONS")
print("=" * 60)

print("Theory:")
print("-------")
print("• Morphology: Shape-based image processing")
print("• Erosion: Shrinks objects, removes noise")
print("• Dilation: Expands objects, fills gaps")
print("• Opening: Erosion followed by dilation (removes noise)")
print("• Closing: Dilation followed by erosion (fills gaps)")

# Create binary image for morphology
_, binary_img = cv2.threshold(gray_img, 127, 255, cv2.THRESH_BINARY)

# Define kernel
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))

# Apply morphological operations
erosion = cv2.erode(binary_img, kernel, iterations=1)
dilation = cv2.dilate(binary_img, kernel, iterations=1)
opening = cv2.morphologyEx(binary_img, cv2.MORPH_OPEN, kernel)
closing = cv2.morphologyEx(binary_img, cv2.MORPH_CLOSE, kernel)

display_images([binary_img, erosion, dilation, opening, closing],
               ['Binary', 'Erosion', 'Dilation', 'Opening', 'Closing'])

# Task 5: Text cleaning with morphology
print("\n TASK 5: Text cleaning using morphological operations")

def create_noisy_text():
    """Create sample text with noise"""
    img = np.zeros((100, 400), dtype=np.uint8)
    font = cv2.FONT_HERSHEY_SIMPLEX
    cv2.putText(img, 'IMAGE PROCESSING', (10, 50), font, 1, 255, 2)
    
    # Add noise
    noise = np.random.random(img.shape)
    img[noise < 0.05] = 255  # Salt noise
    img[noise > 0.95] = 0    # Pepper noise
    return img

noisy_text = create_noisy_text()
cleaned_text = cv2.morphologyEx(noisy_text, cv2.MORPH_CLOSE, 
                               cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3)))

display_images([noisy_text, cleaned_text], ['Noisy Text', 'Cleaned Text'])

# =============================================================================
# 8: Chapter 6 - Image Enhancement
# =============================================================================

print("\n" + "=" * 60)
print("CHAPTER 6: IMAGE ENHANCEMENT")
print("=" * 60)

print("Theory:")
print("-------")
print("• Enhancement: Improve visual quality")
print("• Histogram Equalization: Spreads intensity distribution")
print("• CLAHE: Adaptive histogram equalization")
print("• Brightness/Contrast: Linear transformation")

# Create low contrast image
low_contrast = cv2.convertScaleAbs(gray_img, alpha=0.5, beta=30)

# Enhancement techniques
hist_eq = cv2.equalizeHist(low_contrast)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
clahe_eq = clahe.apply(low_contrast)
bright_contrast = cv2.convertScaleAbs(low_contrast, alpha=1.5, beta=20)

display_images([low_contrast, hist_eq, clahe_eq, bright_contrast],
               ['Low Contrast', 'Hist Equalized', 'CLAHE', 'Bright/Contrast'])

# Task 6: Histogram analysis
print("\n TASK 6: Analyze and plot histograms")

def plot_histogram(images, titles):
    """Plot histograms for multiple images"""
    fig, axes = plt.subplots(2, len(images), figsize=(15, 8))
    
    for i, (img, title) in enumerate(zip(images, titles)):
        # Display image
        axes[0, i].imshow(img, cmap='gray')
        axes[0, i].set_title(title)
        axes[0, i].axis('off')
        
        # Display histogram
        axes[1, i].hist(img.ravel(), bins=256, range=[0, 256], alpha=0.7)
        axes[1, i].set_title(f'{title} Histogram')
        axes[1, i].set_xlabel('Pixel Intensity')
        axes[1, i].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()

plot_histogram([low_contrast, hist_eq, clahe_eq], 
               ['Original', 'Hist Eq', 'CLAHE'])

# =============================================================================
# 9: Chapter 7 - Frequency Domain Processing
# =============================================================================

print("\n" + "=" * 60)
print("CHAPTER 7: FREQUENCY DOMAIN PROCESSING")
print("=" * 60)

print("Theory:")
print("-------")
print("• Fourier Transform: Time/Spatial → Frequency domain")
print("• Low frequencies: Smooth areas, general shape")
print("• High frequencies: Edges, noise, fine details")
print("• Filtering in frequency domain: Multiply by mask")

# Fourier Transform
def apply_fft(img):
    """Apply FFT and return magnitude spectrum"""
    f_transform = np.fft.fft2(img)
    f_shift = np.fft.fftshift(f_transform)
    magnitude_spectrum = np.log(np.abs(f_shift) + 1)
    return f_shift, magnitude_spectrum

f_shift, magnitude = apply_fft(gray_img)

# Create frequency domain filters
def create_filter_mask(shape, filter_type, cutoff):
    """Create filter mask for frequency domain"""
    rows, cols = shape
    crow, ccol = rows//2, cols//2
    
    if filter_type == 'lowpass':
        mask = np.zeros((rows, cols), np.float32)
        mask[crow-cutoff:crow+cutoff, ccol-cutoff:ccol+cutoff] = 1
    else:  # highpass
        mask = np.ones((rows, cols), np.float32)
        mask[crow-cutoff:crow+cutoff, ccol-cutoff:ccol+cutoff] = 0
    
    return mask

# Apply low-pass filter
lowpass_mask = create_filter_mask(gray_img.shape, 'lowpass', 30)
f_shift_lp = f_shift * lowpass_mask
f_ishift_lp = np.fft.ifftshift(f_shift_lp)
img_lp = np.fft.ifft2(f_ishift_lp)
img_lp = np.abs(img_lp)

# Apply high-pass filter
highpass_mask = create_filter_mask(gray_img.shape, 'highpass', 30)
f_shift_hp = f_shift * highpass_mask
f_ishift_hp = np.fft.ifftshift(f_shift_hp)
img_hp = np.fft.ifft2(f_ishift_hp)
img_hp = np.abs(img_hp)

display_images([gray_img, magnitude, img_lp, img_hp],
               ['Original', 'FFT Spectrum', 'Low-pass', 'High-pass'])

# Task 7: Custom frequency domain filtering
print("\n TASK 7: Design custom frequency filters")

def gaussian_filter_freq(shape, sigma):
    """Create Gaussian filter in frequency domain"""
    rows, cols = shape
    crow, ccol = rows//2, cols//2
    
    y, x = np.ogrid[:rows, :cols]
    mask = np.exp(-((x - ccol)**2 + (y - crow)**2) / (2 * sigma**2))
    return mask

# Apply Gaussian low-pass filter
gaussian_mask = gaussian_filter_freq(gray_img.shape, 30)
f_shift_gauss = f_shift * gaussian_mask
f_ishift_gauss = np.fft.ifftshift(f_shift_gauss)
img_gauss = np.abs(np.fft.ifft2(f_ishift_gauss))

display_images([lowpass_mask, gaussian_mask, img_lp, img_gauss],
               ['Ideal LP Filter', 'Gaussian LP Filter', 'Ideal Result', 'Gaussian Result'])

# =============================================================================
# 10: Chapter 8 - Advanced Applications
# =============================================================================

print("\n" + "=" * 60)
print("CHAPTER 8: ADVANCED APPLICATIONS")
print("=" * 60)

# Application 1: Document Scanner
print("Application 1: Document Scanner")
print("------------------------------")

def document_scanner(img):
    """Simulate document scanning pipeline"""
    # 1. Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    
    # 2. Apply Gaussian blur
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # 3. Apply adaptive thresholding
    thresh = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                  cv2.THRESH_BINARY, 11, 2)
    
    # 4. Morphological operations to clean up
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    cleaned = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    
    return gray, thresh, cleaned

# Apply document scanner
gray_doc, thresh_doc, clean_doc = document_scanner(sample_img)
display_images([sample_img, gray_doc, thresh_doc, clean_doc],
               ['Original', 'Grayscale', 'Thresholded', 'Cleaned'])

# Application 2: Feature Detection
print("\nApplication 2: Feature Detection")
print("--------------------------------")

def detect_corners(img):
    """Detect corners using Harris corner detection"""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    corners = cv2.cornerHarris(gray, 2, 3, 0.04)
    
    # Dilate corner points for visualization
    corners = cv2.dilate(corners, None)
    
    # Create result image
    result = img.copy() if len(img.shape) == 3 else cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    result[corners > 0.01 * corners.max()] = [0, 0, 255]
    
    return corners, result

corners, corner_img = detect_corners(sample_img)
display_images([sample_img, corners, corner_img],
               ['Original', 'Corner Response', 'Detected Corners'])

# =============================================================================
# 11: Performance Comparison
# =============================================================================

print("\n" + "=" * 60)
print("PERFORMANCE COMPARISON")
print("=" * 60)

import time

def benchmark_operations(img):
    """Benchmark different operations"""
    operations = {
        'Gaussian Blur': lambda: cv2.GaussianBlur(img, (15, 15), 0),
        'Median Blur': lambda: cv2.medianBlur(img, 15),
        'Bilateral Filter': lambda: cv2.bilateralFilter(img, 15, 80, 80),
        'Canny Edge': lambda: cv2.Canny(img, 50, 150),
        'Histogram Eq': lambda: cv2.equalizeHist(img),
        'Morphology': lambda: cv2.morphologyEx(img, cv2.MORPH_OPEN, 
                                              cv2.getStructuringElement(cv2.MORPH_RECT, (5,5)))
    }
    
    results = {}
    for name, operation in operations.items():
        start_time = time.time()
        for _ in range(10):  # Run 10 times for average
            _ = operation()
        end_time = time.time()
        results[name] = (end_time - start_time) / 10 * 1000  # Convert to ms
    
    return results

# Benchmark with grayscale image
benchmark_results = benchmark_operations(gray_img)

print("Operation Performance (Average time in milliseconds):")
print("-" * 50)
for operation, time_ms in sorted(benchmark_results.items(), key=lambda x: x[1]):
    print(f"{operation:<20}: {time_ms:.2f} ms")

# =============================================================================
#  12: Practical Exercises
# =============================================================================

print("\n" + "=" * 60)
print("PRACTICAL EXERCISES")
print("=" * 60)

print("EXERCISE 1: Image Quality Assessment")
print("--------------------------------------")

def calculate_metrics(img1, img2):
    """Calculate image quality metrics"""
    # Mean Squared Error
    mse = np.mean((img1.astype(float) - img2.astype(float)) ** 2)
    
    # Peak Signal-to-Noise Ratio
    if mse == 0:
        psnr = float('inf')
    else:
        max_pixel = 255.0
        psnr = 20 * np.log10(max_pixel / np.sqrt(mse))
    
    # Structural Similarity Index (simplified)
    mean1, mean2 = np.mean(img1), np.mean(img2)
    var1, var2 = np.var(img1), np.var(img2)
    covar = np.mean((img1 - mean1) * (img2 - mean2))
    
    ssim = ((2 * mean1 * mean2) * (2 * covar)) / ((mean1**2 + mean2**2) * (var1 + var2))
    
    return mse, psnr, ssim

# Compare original with filtered versions
gaussian_blur = cv2.GaussianBlur(gray_img, (15, 15), 0)
mse, psnr, ssim = calculate_metrics(gray_img, gaussian_blur)

print(f"Original vs Gaussian Blur:")
print(f"MSE: {mse:.2f}")
print(f"PSNR: {psnr:.2f} dB")
print(f"SSIM: {ssim:.4f}")

print("\n EXERCISE 2: Build Custom Image Filter")
print("----------------------------------------")

def custom_artistic_filter(img):
    """Create artistic effect combining multiple operations"""
    # Step 1: Apply bilateral filter for smoothing
    smooth = cv2.bilateralFilter(img, 15, 80, 80)
    
    # Step 2: Detect edges
    edges = cv2.Canny(img, 50, 150)
    
    # Step 3: Dilate edges to make them thicker
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    edges = cv2.dilate(edges, kernel, iterations=1)
    
    # Step 4: Convert edges to 3-channel for masking
    edges_3ch = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
    
    # Step 5: Combine smooth image with edge mask
    artistic = smooth.copy()
    artistic[edges_3ch > 128] = 0  # Make edges black
    
    return smooth, edges, artistic

if len(sample_img.shape) == 3:
    smooth, edges, artistic = custom_artistic_filter(sample_img)
    display_images([sample_img, smooth, edges, artistic],
                   ['Original', 'Smoothed', 'Edges', 'Artistic'])

# =============================================================================
# 13: Summary and Best Practices
# =============================================================================

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)

print(" Key Concepts :")
print("------------------------")
print(" Image representation as NumPy arrays")
print(" Color space conversions (RGB, HSV, YCrCb, Grayscale)")
print(" Filtering techniques (Gaussian, Median, Bilateral)")
print(" Edge detection methods (Sobel, Laplacian, Canny)")
print(" Morphological operations (Erosion, Dilation, Opening, Closing)")
print(" Image enhancement (Histogram Equalization, CLAHE)")
print(" Frequency domain processing (FFT, Filtering)")
print(" Practical applications and performance considerations")


print("\n Common Parameter Guidelines:")
print("-------------------------------")
print("• Gaussian Blur: kernel_size = 2*sigma + 1 (approximately)")
print("• Canny Edge: low_threshold = 0.5 * high_threshold")
print("• Morphology: kernel_size depends on feature size")
print("• CLAHE: clip_limit = 2-4 for most natural images")

print("\n Next Steps:")
print("--------------")
print("• Explore advanced topics: Machine Learning, Deep Learning")
print("• Practice with real-world datasets")
print("• Implement custom algorithms")


print("\n" + "=" * 60)

print("=" * 60)